<a href="https://colab.research.google.com/github/mf1060/GenAI/blob/main/HW4/HW4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW 4

Michael Furey

Dr. Forouraghi

CSC 688

3/17/2026


The following notebook discusses the results for Homework #4. This assignment is focused on observing responses with different methods for retrieval augmented generation. This notebook uses models from the [Advanced RAG techniques notebook](https://github.com/bforoura/GENAI26/blob/main/Module3/Advanced_RAG_Techniques.ipynb).

I used the following [tutorial](https://www.geeksforgeeks.org/html/markdown-tables/) for making markdown tables as well as this [reference sheet](https://www.markdownlang.com/cheatsheet/).

In [1]:
!pip install -qU langchain-core langchain-community langchain-google-genai langchain-classic
!pip install -qU faiss-cpu pypdf rank_bm25 flashrank

# Standard imports for environment management
import os
from google.colab import userdata

# API Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.4/333.4 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 78.1 MB/s eta 0:00:00


# **Downloading Files**

To conduct Assignment 4, this notebook uses ten files from the [Congressional Research Service](https://www.congress.gov/crs-products). These are the most recent "In Focus" articles on legislation. Each of these documents are short, between 3-4 pages.

The following recursively splits documents with a 500 character chunk length and a 10% overlap.


# Embedding Chunks in a Vector Store

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

# Updated list to match your specific filenames
# Updated list to match your specific filenames
file_names = [
    "https://www.congress.gov/crs_external_products/IF/PDF/IF10538/IF10538.11.pdf",
    "https://www.congress.gov/crs_external_products/IF/PDF/IF11141/IF11141.6.pdf",
    "https://www.congress.gov/crs_external_products/IF/PDF/IF12310/IF12310.4.pdf",
    "https://www.congress.gov/crs_external_products/IN/PDF/IN12222/IN12222.6.pdf",
    "https://www.congress.gov/crs_external_products/IN/PDF/IN12364/IN12364.2.pdf",
    "https://www.congress.gov/crs_external_products/IN/PDF/IN12198/IN12198.3.pdf",
    "https://www.congress.gov/crs_external_products/IN/PDF/IN12575/IN12575.3.pdf",
    "https://www.congress.gov/crs_external_products/IF/PDF/IF13116/IF13116.4.pdf",
    "https://www.congress.gov/crs_external_products/IF/PDF/IF13168/IF13168.1.pdf",
    "https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf",
]

all_docs = []

# Loop through each file, load its content, and split it into chunks
for file in file_names:
    try:
        print(f"Processing {file}...")
        loader = PyPDFLoader(file)

        # Recursive splitter attempts to keep paragraphs and sentences together
        # chunk_size is total characters; overlap keeps context across boundaries
        splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=15)
        chunks = loader.load_and_split(splitter)
        all_docs.extend(chunks)
    except:
        print(f"Warning: '{file}' not found in the sidebar. Please upload it.")

if all_docs:
    print(f"\nSuccess! Total text chunks created: {len(all_docs)}")

Processing https://www.congress.gov/crs_external_products/IF/PDF/IF10538/IF10538.11.pdf...
Processing https://www.congress.gov/crs_external_products/IF/PDF/IF11141/IF11141.6.pdf...
Processing https://www.congress.gov/crs_external_products/IF/PDF/IF12310/IF12310.4.pdf...
Processing https://www.congress.gov/crs_external_products/IN/PDF/IN12222/IN12222.6.pdf...
Processing https://www.congress.gov/crs_external_products/IN/PDF/IN12364/IN12364.2.pdf...
Processing https://www.congress.gov/crs_external_products/IN/PDF/IN12198/IN12198.3.pdf...
Processing https://www.congress.gov/crs_external_products/IN/PDF/IN12575/IN12575.3.pdf...
Processing https://www.congress.gov/crs_external_products/IF/PDF/IF13116/IF13116.4.pdf...
Processing https://www.congress.gov/crs_external_products/IF/PDF/IF13168/IF13168.1.pdf...
Processing https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf...

Success! Total text chunks created: 245


In [3]:
import time
from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_exception_type
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# Initialize Gemini Embeddings (2026 Stable Model)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document"
)



# Define the Retry-Safe Embedding Wrapper
# This function will wait and retry automatically if we get a 429 error.
@retry(
    wait=wait_random_exponential(min=1, max=60),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(Exception) # Catch the GoogleGenerativeAIError
)
def embed_with_retry(vector_store, batch):
    if vector_store is None:
        return FAISS.from_documents(batch, embeddings)
    else:
        vector_store.add_documents(batch)
        return vector_store



# Process with Batching and Retries
batch_size = 15 # Smaller batches help prevent hitting the limit too fast
vectorstore = None

print(f"Embedding {len(all_docs)} chunks with rate-limit protection...")
for i in range(0, len(all_docs), batch_size):
    batch = all_docs[i : i + batch_size]
    try:
        vectorstore = embed_with_retry(vectorstore, batch)
        print(f" Processed {i + len(batch)}/{len(all_docs)}...")
    except Exception as e:
        print(f" Failed after retries: {e}")
        break
    time.sleep(3) # A steady 3-second heartbeat to keep the API happy





Embedding 245 chunks with rate-limit protection...
 Processed 15/245...
 Processed 30/245...
 Processed 45/245...
 Processed 60/245...
 Processed 75/245...
 Processed 90/245...
 Processed 105/245...
 Processed 120/245...
 Processed 135/245...
 Processed 150/245...
 Processed 165/245...
 Processed 180/245...
 Processed 195/245...
 Processed 210/245...
 Processed 225/245...
 Processed 240/245...
 Processed 245/245...


## Function for Creating Retrievers

The following is a function for creating retrievers for Hybrid, MMR, and Naive RAG techniques with the following [code snippets](https://github.com/bforoura/GENAI26/blob/main/Module3/Slides/avanced_rag_notebook.html).

In [4]:
# This is a new function to create a retriever
def create_retreiver(all_docs, retriever_name, embeddings):

    if (retriever_name == "Hybrid"):
      vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5, "fetch_k": 20})
      bm25_retriever = BM25Retriever.from_documents(all_docs)
      bm25_retriever.k = 5
      print("\n Hybrid Retriever is ready!")

      retriever = EnsembleRetriever(
        retrievers=[vector_retriever, bm25_retriever],
        weights=[0.5, 0.5]
    )

    elif (retriever_name == "MMR"):
      retriever = vectorstore.as_retriever(
          search_type="mmr",
          search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5})
      print("\n MMR Retriever is ready!")

    else:
      retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
      print("\n Naive Retriever is ready!")

    return retriever

# Creating Retreivers

In [5]:
mmr = create_retreiver(all_docs, "MMR", embeddings)
hybrid = create_retreiver(all_docs, "Hybrid", embeddings)
naive = create_retreiver(all_docs, "Naive", embeddings)


 MMR Retriever is ready!

 Hybrid Retriever is ready!

 Naive Retriever is ready!


# Functions for Naive, MMR, and Hybrid Retrievers

In [6]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# Hub import for modular LangChain
try:
    from langchain_classic import hub
except ImportError:
    import langchainhub as hub


# UPDATED 2026 IMPORTS
try:
    # Most stable 2026 path for Contextual Compression
    from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
    from langchain.retrievers.document_compressors.flashrank_rerank import FlashrankRerank
except ImportError:
    # Alternative path used in some v1.x sub-versions
    from langchain_classic.retrievers import ContextualCompressionRetriever
    from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank


# Function for Retrieving Documents with an Option for Reranking

In [7]:
def retrieve_docs(retriever, query, test_name, retreiver_type, re_rank=False):

    print(f"\n--- Testing: {test_name}, Method: {retreiver_type} ---")
    print(f"Query: {query}")

    docs=[]

    if (re_rank == True):
      # Initialize FlashRank (The tiny but mighty re-ranker)
      # We set top_n=3 to ensure the LLM only sees the most relevant facts.
      compressor = FlashrankRerank(model="ms-marco-MiniLM-L-12-v2", top_n=3)

      # Create the Compression Retriever
      # This wraps the Hybrid Retriever we just successfully built.
      compression_retriever = ContextualCompressionRetriever(
          base_compressor=compressor,
          base_retriever=retriever
      )

      docs = compression_retriever.invoke(query)

    else:
      docs = retriever.invoke(query)

    for i, doc in enumerate(docs[:3]): # Show top 3 results
        source = doc.metadata.get('source', 'Unknown')
        # We truncate the content for readability
        content = doc.page_content[:200].replace('\n', ' ')
        print(f"Rank {i+1}: [{source}] \n {content}...")

    # Gemini 2.5 Flash is the 'sweet spot' for speed and availability.
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.1
    )


    # Attach Rate-Limit Protection
    llm_with_retry = llm.with_retry(
        stop_after_attempt=5,
        wait_exponential_jitter=True
    )


    # Pull the standard RAG prompt
    prompt = hub.pull("rlm/rag-prompt")


    # THE LCEL PIPELINE (The "Pipe" Syntax)
    rag_chain = (
        RunnableParallel({
            "context": retriever,
            "question": RunnablePassthrough()
        })
        | prompt
        | llm_with_retry
        | StrOutputParser()
    )

    # Final Execution
    print("Generative Answer")

    try:
        answer = rag_chain.invoke(query)
        print(f"QUESTION: {query}")
        print(f"\nANSWER:\n{answer}")
    except Exception as e:
        print(f" Chain failed: {e}")

# Function for Testing Each Trial

Each trial asks three questions. The first is a factual question that asks about the Fourth Amendment. The second is a conceptual question that asks about options for Congress to legislate. The third is a technical question designed to ask about the term, "appropriation" which is legislation that sets money aside for certain programs.

In [8]:
from types import MethodType
def test_trial(retreiver, method, re_rank=False):

  ## Question 1
  retrieve_docs(retreiver, "What is the Fourth Amendment?", "Factual Question", method, re_rank)

  ## Question 2
  retrieve_docs(retreiver, "How can Congress protect privacy rights?", "Conceptual Question", method, re_rank)

  ## Question 3
  retrieve_docs(retreiver, "What is an appropriation?", "Technical Question", method, re_rank)


# Testing Queries (No-Re Rank)

The following code tests these three queries using different techniques of retreival augmented generation. The first, Trial A, uses Naive RAG. The second, Trial B, uses MMR. The third, Trial C, uses a Hybrid approach.

## Trail A

In [9]:
test_trial(naive, "Naive", re_rank=False)


--- Testing: Factual Question, Method: Naive ---
Query: What is the Fourth Amendment?
Rank 1: [https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf] 
 The Fourth Amendment dictates that search warrants must  “particularly describ[e] the place to be searched, and the  persons or things to be seized.” On this score, the Court has  said that the “requi...
Rank 2: [https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf] 
 https://crsreports.congress.gov    February 26, 2026 Fourth Amendment Search Warrant Requirements The Fourth Amendment prohibits unreasonable searches  and seizures. As a result, law enforcement gener...
Rank 3: [https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf] 
 Fourth Amendment Search Warrant Requirements  https://crsreports.congress.gov | IF13169 · VERSION 1 · NEW      Disclaimer  This document was prepared by the Congressional Research Service (CRS). CRS s...
Generative Answer
QUESTION: What is t

The Naive RAG approach mostly pulls chunks from the same source, except for the last question. Even though the last question pulled in a variety of sources, they were all similar chunks (a disclaimer in the Congressional Research Service documentation).

## Trial B


In [10]:
test_trial(mmr, "MMR", re_rank=False)


--- Testing: Factual Question, Method: MMR ---
Query: What is the Fourth Amendment?
Rank 1: [https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf] 
 The Fourth Amendment dictates that search warrants must  “particularly describ[e] the place to be searched, and the  persons or things to be seized.” On this score, the Court has  said that the “requi...
Rank 2: [https://www.congress.gov/crs_external_products/IN/PDF/IN12198/IN12198.3.pdf] 
 permission of the copyright holder if you wish to copy or otherwise use copyrighted material....
Rank 3: [https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf] 
 has observed that this “exclusionary rule” is a “judicially  created remedy designed to safeguard Fourth Amendment  rights generally through its deterrent effect, rather than a  personal constitutiona...
Generative Answer
QUESTION: What is the Fourth Amendment?

ANSWER:
The provided context indicates that the Fourth Amendment dictates that search w

MMR appears to have stronger results for defining "appropriation" in the last question. Although the model admitted to not knowing the answer, it did find chunks from more varied sources to explain what an appropriation is. This is particularly the case with chunk 2 for question 3. The answer was actually close to defining what an appropriation is. I believe that this is because it drew from a few more varied sources. Naive RAG only found chunks related to a disclaimer in the Congressional Research Service material.

## Trial C

In [11]:
test_trial(hybrid, "Hybrid", re_rank=False)


--- Testing: Factual Question, Method: Hybrid ---
Query: What is the Fourth Amendment?
Rank 1: [https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf] 
 https://crsreports.congress.gov    February 26, 2026 Fourth Amendment Search Warrant Requirements The Fourth Amendment prohibits unreasonable searches  and seizures. As a result, law enforcement gener...
Rank 2: [https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf] 
 searched, and the persons or things to be seized.” This In  Focus provides examples of judicial interpretations of this  provision regarding what is required from law enforcement  seeking to obtain a ...
Rank 3: [https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf] 
 The Fourth Amendment dictates that search warrants must  “particularly describ[e] the place to be searched, and the  persons or things to be seized.” On this score, the Court has  said that the “requi...
Generative Answer
QUESTION: What is 

Interestingly, the hybrid approach does not give us a better answer for "appropriation" even though it is a technical term. In fact, MMR gave us the most responsive chunks for Question 3. It is clear that this is ranking documents using a keyword search, because Question 1 which asks about the Fourth Amendment, is pulling chunks all from the document on the Fourth Amendment.

# Analysis

## Naive

| Trial | Score |	Result |
|---| --- | --- |
|A	| 4| The Fourth Amendment prohibits unreasonable searches and seizures. It dictates that search warrants must be issued only upon probable cause, supported by oath or affirmation. Warrants must also particularly describe the place to be searched and the persons or things to be seized.|
|B	| 5|Congress can protect privacy rights by enacting specific legislation, such as the Privacy Protection Act of 1980, which limits law enforcement's ability to search or seize news media materials. It can also pass laws like the Gramm-Leach-Bliley Act to govern data privacy in sectors such as the financial system. Additionally, Congress can augment existing protections, like those under the Fourth Amendment, by imposing stricter protocols for wiretap warrant applications.|
|C	| 2	| I don't know the answer. The provided context does not contain information about what an appropriation is; it only discusses copyrighted material.

## MMR

| Trial | Score |	Result |
|---| --- | --- |
|A	| 3 |The provided context indicates that the Fourth Amendment dictates that search warrants must particularly describe the place to be searched and the persons or things to be seized. It also mentions that the exclusionary rule is a remedy designed to safeguard Fourth Amendment rights. However, the context does not provide a general definition of what the Fourth Amendment is.|
|B	| 4	|Congress can protect privacy rights by enacting legislation, such as establishing protocols for wiretap warrant applications. For example, Congress passed the Privacy Protection Act of 1980 to limit law enforcement's ability to search or seize news media materials. Additionally, Congress enacted the Gramm-Leach-Bliley Act (GLBA) to safeguard personal financial information.|
|C	|4	| I don't know the answer. The provided context mentions "appropriations" in relation to government funding and legislative committees, but it does not define what an appropriation is.|

## Hybrid

| Trial | Score |	Result |
|---| --- | --- |
|A	| 5 |The Fourth Amendment prohibits unreasonable searches and seizures by law enforcement. Generally, it requires officers to obtain a search warrant from a neutral magistrate before entering a private space to look for evidence. Warrants must be based on probable cause, supported by oath or affirmation, and particularly describe the place to be searched and the items to be seized.|  
|B	|5	| Congress can protect privacy rights by enacting specific legislation, such as the Privacy Protection Act of 1980, which limits law enforcement's ability to search or seize certain news media materials. They can also impose stricter protocols for wiretap warrant applications, thereby augmenting Fourth Amendment protections. Additionally, Congress has passed laws like the Gramm-Leach-Bliley Act (GLBA) to govern data privacy in the financial system, requiring institutions to develop privacy policies and offer opt-out options for personal financial information.|
|C	| 3	| I don't know the answer. The provided context does not contain information about what an appropriation is.|


**Overall Performance**
I think that Naive RAG and Hybrid did a good job with the first question specifically. Because both of these models are particularly grounded, they can pull information well for explaining a common term, such as the Fourth Amendment. Naive responses are particularly grounded with semantic context while Hybrid responses are grounded with keywords. I think that Naive RAG and Hybrid also did a great job with question #2 for similar reasons. MMR answered question #3 best of all three models even though it admitted to not knowing the answer. I tended to score responses lower if they were less narrative.

I think that MMR performed the best on average, because it provided all answers that were helpful to the questions. I think that pulling from a variety of sources was helpful to answering the questions. MMR was particularly helpful for pulling from a diverse set of sources.

**Compare Trial A to Trial B. Did the Naive approach return three chunks that all said the same thing? Did MMR provide a more "complete" picture for your conceptual question?**

Naive RAG tended to return more chunks from the same document. MMR tended to provide more results that ultimately helped answer the question. For instance, MMR was able to draw in some chunks that could explain what an appropriation is. I think that MMR was helpful for making a more complete picture. I think that MMR would be helpful for questions that are not answered by a document on-point.

**In Trial C, did the Hybrid search find information that the Vector-only searches (A and B) missed? Hint. Look for specific dates, names, or technical IDs in your documents.**

I believe that the Hybrid search pulled in more specific terms to answer Question 1, making it a better answer with more detail. For instance, it not only described the Fourth Amendment but gave further explanation to discuss private and public places.

**If a retriever failed to find the answer, did the LLM admit it didn't know, or did it use its internal knowledge to "guess"? Explain how retrieval quality directly impacts "Groundedness."**

All models admitted that they did not know the answer for Question 3. MMR extrapolated a little more to add in context when the word "appropriation" was used in a chunk. MMR made the model less grounded, but it created a better response. This likely because it pulled in more documents to bring context to the word, "appropriation" and provide some response for how it was used.

**Which model would I put in production?**

I would probably use the MMR model for production with less documents and then the Hybrid model for more documents. I think that the MMR model does a particularly good job drawing in other sources, but I think that the Hybrid model would be helpful for pulling keywords for a variety of sources.








# Extra Credit

The following tests each question with a re-rank. Each of these trials had similar results to the ones without a re-ranker.

## Trial A

In [12]:
test_trial(naive, "Naive", re_rank=True)


--- Testing: Factual Question, Method: Naive ---
Query: What is the Fourth Amendment?


ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:00<00:00, 31.9MiB/s]


Rank 1: [https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf] 
 The Fourth Amendment dictates that search warrants must  “particularly describ[e] the place to be searched, and the  persons or things to be seized.” On this score, the Court has  said that the “requi...
Rank 2: [https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf] 
 https://crsreports.congress.gov    February 26, 2026 Fourth Amendment Search Warrant Requirements The Fourth Amendment prohibits unreasonable searches  and seizures. As a result, law enforcement gener...
Rank 3: [https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf] 
 searched, and the persons or things to be seized.” This In  Focus provides examples of judicial interpretations of this  provision regarding what is required from law enforcement  seeking to obtain a ...
Generative Answer
QUESTION: What is the Fourth Amendment?

ANSWER:
The Fourth Amendment prohibits unreasonable searches and 

## Trial B

In [13]:
test_trial(mmr, "MMR", re_rank=True)


--- Testing: Factual Question, Method: MMR ---
Query: What is the Fourth Amendment?
Rank 1: [https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf] 
 The Fourth Amendment dictates that search warrants must  “particularly describ[e] the place to be searched, and the  persons or things to be seized.” On this score, the Court has  said that the “requi...
Rank 2: [https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf] 
 Fourth Amendment Search Warrant Requirements  https://crsreports.congress.gov | IF13169 · VERSION 1 · NEW      Disclaimer  This document was prepared by the Congressional Research Service (CRS). CRS s...
Rank 3: [https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf] 
 has observed that this “exclusionary rule” is a “judicially  created remedy designed to safeguard Fourth Amendment  rights generally through its deterrent effect, rather than a  personal constitutiona...
Generative Answer
QUESTION: What is the

## Trial C


In [14]:
test_trial(hybrid, "Hybrid", re_rank=True)


--- Testing: Factual Question, Method: Hybrid ---
Query: What is the Fourth Amendment?
Rank 1: [https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf] 
 The Fourth Amendment dictates that search warrants must  “particularly describ[e] the place to be searched, and the  persons or things to be seized.” On this score, the Court has  said that the “requi...
Rank 2: [https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf] 
 https://crsreports.congress.gov    February 26, 2026 Fourth Amendment Search Warrant Requirements The Fourth Amendment prohibits unreasonable searches  and seizures. As a result, law enforcement gener...
Rank 3: [https://www.congress.gov/crs_external_products/IF/PDF/IF13169/IF13169.1.pdf] 
 searched, and the persons or things to be seized.” This In  Focus provides examples of judicial interpretations of this  provision regarding what is required from law enforcement  seeking to obtain a ...
Generative Answer
QUESTION: What is 

# Analysis

## Naive

| Trial | Score |	Result |
|---| --- | --- |
|A	| 4| The Fourth Amendment prohibits unreasonable searches and seizures. It dictates that law enforcement generally must obtain a search warrant from a neutral magistrate before entering a private space to look for evidence. Warrants must be based on probable cause, supported by oath or affirmation, and particularly describe the place to be searched and the persons or things to be seized.|
|B	| 4| Congress can protect privacy rights by enacting legislation, such as the Privacy Protection Act of 1980, which limits law enforcement's ability to search or seize certain materials. They can also augment Fourth Amendment protections by imposing stricter protocols for wiretap warrant applications. Additionally, Congress has passed acts like the Gramm-Leach-Bliley Act to govern data privacy in specific sectors, requiring privacy policies and opt-out options.|
|C	| 2	| I apologize, but the provided context does not contain information about what an appropriation is. The documents primarily discuss copyright material and permissions.|

## MMR

| Trial | Score |	Result |
|---| --- | --- |
|A	| 3 | The provided context states that the Fourth Amendment dictates search warrants must "particularly describ[e] the place to be searched, and the persons or things to be seized." It also mentions that the exclusionary rule is designed to safeguard Fourth Amendment rights. However, the context does not provide a general definition of what the Fourth Amendment is.|
|B	| 4	| Congress can protect privacy rights by enacting legislation, such as the Privacy Protection Act of 1980, which limits law enforcement's ability to search or seize news media. Additionally, Congress can establish protocols for specific actions like wiretap warrant applications. They can also pass acts like the Gramm-Leach-Bliley Act (GLBA) to safeguard personal financial information.|
|C	|4	| I don't know the answer. The provided context mentions "appropriations" in relation to funding and legislative actions by committees, but it does not define what an appropriation is.|

## Hybrid

| Trial | Score |	Result |
|---| --- | --- |
|A	| 3| The Fourth Amendment prohibits unreasonable searches and seizures. It generally requires law enforcement to obtain a search warrant from a neutral magistrate before entering a private space to look for evidence. Warrants must be based on probable cause, supported by oath or affirmation, and particularly describe the place to be searched and the items to be seized.|  
|B	|	4| Congress can protect privacy rights by enacting specific legislation, such as the Privacy Protection Act of 1980, which limits law enforcement's ability to search or seize certain news media. They can also augment Fourth Amendment protections by imposing stricter protocols for wiretap warrant applications. Additionally, Congress has passed laws like the Gramm-Leach-Bliley Act to govern data privacy in the financial system, requiring institutions to develop privacy policies and offer opt-out options for personal financial information.|
|C	| 4	| I don't know the answer to your question based on the provided context. The documents discuss topics such as copyrighted material, family and medical leave, data privacy, and financial institution regulations, but they do not define "appropriation."|

Although there was some difference in responses, I did not observe a noticeable change with regards to how the re-ranker set the order of chunks.

**Did the Reranker move a more relevant chunk from the "bottom" of the list to the "top"?**

Although there was some difference in responses with a re-ranking, I did not directly observe whether a more relevant chunk was moved from the bottom to the top of the ranking. It's possible that the questions here were more specific to one document over another and a re-ranking may not make a very large difference.

**Explain the trade-off: Is the extra latency, i.e., time taken to rerank, worth the improvement in precision for your specific use case?**

I can imagine that re-ranking would be very helpful to a more complex set of documents with a broader range of questions. I think that the re-ranking here may be less needed when working with a smaller set of documents and more specific questions.
